# Pandas Interview Preparation

Comprehensive, interview-focused reference covering Series/DataFrame creation, IO, indexing & selection, groupby/aggregation, reshaping, time series, performance tips, and common interview exercises with concise examples.

## Setup: import pandas and check versions

In [2]:
import pandas as pd
import numpy as np
print('pandas:', pd.__version__, 'numpy:', np.__version__)


pandas: 3.0.0 numpy: 2.4.2


## Series and DataFrame creation (common patterns)

In [ ]:
s = pd.Series([10,20,30], index=['a','b','c'])
print(s)
df = pd.DataFrame({'A':[1,2,3], 'B':[4,5,6]})
print(df)
arr = np.random.randn(4,3)
df2 = pd.DataFrame(arr, columns=list('XYZ'))
print(df2.head())


## Reading / Writing data (CSV, Excel, Parquet) — concise examples

In [ ]:
# read CSV (specify dtypes for performance)
# df = pd.read_csv('data.csv', dtype={'id':int, 'flag':bool})
# df.to_parquet('data.parquet')
print('I/O examples: pd.read_csv / df.to_parquet')


## Inspecting data: head, tail, info, describe

In [ ]:
print(df2.head())
print(df2.info())
print(df2.describe())


## Indexing and selection: `loc`, `iloc`, boolean indexing, `at`/`iat`

In [ ]:
df = pd.DataFrame({'name':['a','b','c','d'], 'score':[9,8,7,10], 'group':['x','y','x','y']})
print(df.loc[0])
print(df.iloc[0:2])
print(df[df['score']>8])
df.at[0,'score'] = 11  # fast scalar set
print(df)


## Adding / transforming columns: vectorized ops, `assign`, `apply`

In [ ]:
df['score2'] = df['score'] * 2  # vectorized
print(df)
df = df.assign(score_norm = lambda d: (d.score - d.score.mean()) / d.score.std())
print(df)
# use apply for row-wise logic (slower); prefer vectorized ops when possible
df['top'] = df['score'].apply(lambda x: x>8)
print(df)


## GroupBy: aggregation, transform, filter, iteration

In [ ]:
g = df.groupby('group')
print(g['score'].mean())
# aggregate multiple functions
print(g.agg({'score':['mean','max'], 'score2':'sum'}))
# transform preserves index (useful for adding columns)
df['group_mean'] = g['score'].transform('mean')
print(df)


## Merging, joining and concatenation

In [ ]:
left = pd.DataFrame({'id':[1,2,3], 'val':['a','b','c']})
right = pd.DataFrame({'id':[2,3,4], 'val2':[10,20,30]})
print(pd.merge(left, right, on='id', how='inner'))
print(pd.concat([left, right], ignore_index=True, sort=False))


## Reshaping: pivot, pivot_table, melt, stack/unstack, wide<->long

In [ ]:
df_long = pd.DataFrame({'name':['a','a','b','b'], 'year':[2020,2021,2020,2021], 'val':[1,2,3,4]})
print(df_long.pivot(index='name', columns='year', values='val'))
print(pd.pivot_table(df_long, index='name', columns='year', values='val', aggfunc='sum'))
print(df_long.melt(id_vars=['name'], value_vars=['val']))


## Missing data: detect, drop, fill, interpolate

In [ ]:
dfm = pd.DataFrame({'A':[1,np.nan,3], 'B':[np.nan,2,3]})
print(dfm.isna())
print(dfm.dropna())
print(dfm.fillna(0))
print(dfm.interpolate())


## Datetime: to_datetime, indexing, resample, shifting

In [ ]:
d = pd.DataFrame({'date':['2020-01-01','2020-01-02','2020-01-05'], 'value':[1,2,3]})
d['date'] = pd.to_datetime(d['date'])
d = d.set_index('date')
print(d.resample('D').asfreq())
print(d.resample('D').ffill())
print(d.shift(1))  # lag by 1


## Categorical dtype, memory and performance tips

In [ ]:
df_cat = pd.DataFrame({'city': ['ny','la','ny','sf','la']})
print(df_cat['city'].astype('category').cat.categories)
# use `dtype` in read_csv for large data to save memory


## Window functions: rolling, expanding, ewm

In [ ]:
ts = pd.Series([1,2,3,4,5])
print(ts.rolling(3).mean())
print(ts.expanding().sum())
print(ts.ewm(alpha=0.5).mean())


## MultiIndex and advanced indexing

In [ ]:
arrays = [[1,1,2,2],[ 'a','b','a','b']]
mi = pd.MultiIndex.from_arrays(arrays, names=('num','let'))
dfm = pd.DataFrame({'val':[10,20,30,40]}, index=mi)
print(dfm.index.names)
print(dfm.loc[(1,'a')])


## Useful pandas functions & helpers (examples)

In [ ]:
# value_counts / normalize / sort_values
s = pd.Series(['a','b','a','c','b','a'])
print('value_counts:', s.value_counts())
print('proportions:', s.value_counts(normalize=True))
# nunique / unique / factorize
print('nunique:', s.nunique(), 'unique:', s.unique())
print('factorize:', pd.factorize(s))
# map / replace / applymap (elementwise on DataFrame)
dfx = pd.DataFrame({'col':['a','b','c'], 'v':[1,2,3]})
print(dfx.replace({'a':'A'}))
print(dfx['col'].map({'a':'A','b':'B'}))
print(dfx.applymap(lambda x: str(x).upper()))


In [ ]:
# query / eval for readable filtering and fast expressions
dfq = pd.DataFrame({'a':[1,2,3], 'b':[10,20,30]})
print(dfq.query('a > 1'))
print(pd.eval('dfq.a + dfq.b'))
# sample / head / tail
print('sample:', dfq.sample(2))
# isin
print(dfq[dfq['a'].isin([1,3])])


In [3]:
# explode (useful when a column contains lists)
dfe = pd.DataFrame({'id':[1,2], 'tags':[['x','y'], ['z']]})
print('explode ->', dfe.explode('tags'))
# get_dummies (one-hot encoding)
print(pd.get_dummies(pd.Series(['a','b','a']), prefix='cat'))
# cut / qcut for binning
nums = pd.Series([0,5,10,15,20])
print(pd.cut(nums, bins=[0,10,20]))
print(pd.qcut(nums, 2))


explode ->    id tags
0   1    x
0   1    y
1   2    z
   cat_a  cat_b
0   True  False
1  False   True
2   True  False
0             NaN
1     (0.0, 10.0]
2     (0.0, 10.0]
3    (10.0, 20.0]
4    (10.0, 20.0]
dtype: category
Categories (2, interval[int64, right]): [(0, 10] < (10, 20]]
0    (-0.001, 10.0]
1    (-0.001, 10.0]
2    (-0.001, 10.0]
3      (10.0, 20.0]
4      (10.0, 20.0]
dtype: category
Categories (2, interval[float64, right]): [(-0.001, 10.0] < (10.0, 20.0]]


In [ ]:
# pct_change / shift / rank / corr / cov
dfp = pd.DataFrame({'val':[100,110,105,115]})
print('pct_change:', dfp['val'].pct_change())
print('shift(1):', dfp['val'].shift(1))
print('rank:', dfp['val'].rank())
print('corr matrix:', dfp.corr())


In [ ]:
# pipe for chaining transformations (readable pipelines)
def add_col(d):
    return d.assign(total=d.sum(axis=1))
piped = pd.DataFrame({'a':[1,2], 'b':[3,4]}).pipe(add_col)
print('pipe result:', piped)
# explode + groupby example: count tags per id
print(dfe.explode('tags').groupby('tags').size())


In [ ]:
# performance helpers: memory_usage, astype optimizations
dfl = pd.DataFrame({'s': ['a']*1000 + ['b']*1000, 'n': list(range(2000))})
print('memory usage (bytes):', dfl.memory_usage(deep=True).sum())
dfl['s'] = dfl['s'].astype('category')
print('after category:', dfl.memory_usage(deep=True).sum())


## Common interview exercises (with hint code)

1. Given a DataFrame of user events, find the top-3 items per user (hint: groupby + apply or using `nlargest`).
2. Pivot hourly data into daily aggregates without explicit loops (hint: `resample`).
3. Given sales data, compute month-over-month growth per product (hint: groupby + pct_change).
4. Remove duplicate rows keeping the latest by timestamp (hint: sort_values + drop_duplicates(subset=..., keep='last')).

In [ ]:
# Example: top-2 per group using groupby + nlargest
df_ex = pd.DataFrame({'user':[1,1,1,2,2], 'item':[10,11,12,10,13], 'score':[5,3,8,7,2]})
top2 = df_ex.groupby('user', group_keys=False).apply(lambda d: d.nlargest(2, 'score'))
print(top2)


## Gotchas and best practices

- Avoid Python loops over rows; prefer vectorized operations or `apply` only when necessary.
- Be explicit with dtypes when loading large CSVs to reduce memory usage.
- Use `inplace=False` patterns (assignment) for clearer code; `inplace=True` is rarely faster.
- When performance matters, consider using `pd.eval`, `numexpr`, or switching to Dask for out-of-core data.

---
**Next steps:** Run example cells in your environment to verify outputs and I can add additional interview-style exercises (NLP/time-series/joins optimizations) on request.